# Alıştırmalar: Function Calling & Tooling

## Genel Bakış

Pydantic şemaları kullanarak tip güvenli (type-safe) araçlar oluşturmayı, tam araç çalıştırma (tool execution) desenini uygulamayı ve yapay zeka yeteneklerini genişleten çoklu araç sistemleri geliştirme pratiği.


---

## Ödev1: Weather Tool with Complete Execution Loop (Tam Çalıştırma Döngüsüne Sahip Hava Durumu Aracı) ⛅

**Amaç**: Bir hava durumu aracı oluşturun ve tam 3 adımlı araç çalıştırma desenini uygulayın: (generate → execute → respond).

**Görevler**:

1. Aşağıdakileri kabul eden Pydantic schema'ya sahip bir weather tool build edin:

   - `city` (string, required) - City name
   - `units` (Literal["celsius", "fahrenheit"], optional, default: "fahrenheit") - Temperature unit

2. Tool'un en az 5 city için simulated weather data döndürmesini implement edin.

3. Complete 3-step execution pattern'ı implement edin:

   - **Step 1**: LLM'den tool call alın
   - **Step 2**: Tool'u execute edin
   - **Step 3**: Final response için result'ı LLM'e geri gönderin

4. Different cities ve units kullanarak multiple queries ile test edin.

**Test Senaryoları**:
- "What's the weather in Tokyo?"
- "Tell me the temperature in Paris in celsius"
- "Is it raining in London?"

**Başarı Kriterleri**:
- Araç, uygun bir Pydantic şeması kullanmalı.
- Parametrelerde Field(description=...) kullanılmalı.
- Celsius ve Fahrenheit desteklenmeli.
- 3 adımlı çalıştırma süreci eksiksiz uygulanmalı.
- LLM araç sonucunu kullanarak doğal dilde cevap üretmeli.
- Çıktıda her adım açık şekilde gösterilmeli.


**İpuçları**:

In [ ]:
# 1. Gerekli modülleri yükleyin.
!pip install langchain-openai

In [3]:
# 1. Gerekli modülleri import edin
import os
from typing import Literal, Optional
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# 2. Pydantic BaseModel ile input schema oluşturun
#    - Field(description="...") ile city: str
#    - default değer ile units: Optional[Literal["celsius", "fahrenheit"]]

# 3. @tool decorator kullanarak bir weather tool oluşturun:
#    - Pydantic modelini belirtmek için args_schema parameter kullanın
#    - simulated weather data döndürecek function implemente edin
#    - tool description olarak detaylı docstring ekleyin

# 4. model.bind_tools() kullanarak tool'u modele bind edin

# 5. 3-step execution pattern implemente edin:
#    Step 1: user query ile model'i invoke edin, tool_calls olup olmadığını kontrol edin
#    Step 2: tool.invoke(tool_call["args"]) ile tool'u execute edin
#    Step 3: HumanMessage, AIMessage ve ToolMessage ile messages list oluşturun
#            Ardından final natural language response için model'i tekrar invoke edin

## Ödev2: Çoklu Araçlı Seyahat Asistanı (Multi-Tool Travel Assistant) 🌍

**Amaç**:
Birden fazla aracın bulunduğu (multiple tools) ve LLM'in uygun aracı otomatik seçebildiği bir seyahat asistanı geliştirin.

**Görevler**:

1. Üç araç (tool) oluşturun:

   - **Döviz Çevirici (Currency Converter)**: Currency'ler arasında amount convert edin.  
     Desteklenen currency'ler: USD, EUR, GBP, JPY

   - **Mesafe Hesaplayıcı (Distance Calculator)**: İki city arasındaki distance'ı miles veya kilometers olarak hesaplayın.

   - **Saat Dilimi Aracı (Time Zone Tool)**: Bir city'deki current time bilgisini alın ve time difference hesaplayın.

2. Her tool aşağıdakilere sahip olmalıdır:

   - Açık ve Anlaşılır Bir İsim kullanın.
   - Ayrıntılı Docstring: LLM'in hangi durumda bu aracı kullanacağını anlamasını sağlayacak açıklamalar yazın.
   - Pydantic Şeması: Parameter descriptions içeren proper Pydantic schema. Tüm parametrelerde `Field(description="...")` kullanın.

3. Üç tool'un tamamını model'e bind edin.

4. Farklı tool'lar gerektiren queries ile test edin:

   - "Convert 100 USD to EUR"
   - "What's the distance between New York and London?"
   - "What time is it in Tokyo right now?"`
   - "If it's 3pm in Seattle, what time is it in Paris?"

**Başarı Kriterleri**:

- Üç aracın tamamı doğru çalışmalı.
- LLM doğru aracı otomatik seçmeli.
- Araç açıklamaları seçim yapabilmek için yeterince açık olmalı.
- Gerçekçi simüle edilmiş sonuçlar dönmeli.
- Hatalı girişleri yönetebilmeli.

**İpuçları**:

In [ ]:
# 1. Gerekli modules import edin
import os
from typing import Literal, Optional
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# 2. Her tool için Pydantic input schemas oluşturun

# 3. @tool decorator kullanarak üç tool oluşturun:
#    Currency Converter - amount, from_currency, to_currency parameters ile
#    Distance Calculator - from_city, to_city ve units parameters ile
#    Time Zone Tool - city parameter ile
#    Her birinin aşağıdakilere sahip olduğundan emin olun:
#    - Clear, descriptive docstring
#    - Parameters üzerinde Field(description="...") bulunan proper Pydantic schema
#    - Uygun data döndüren simulated implementation

# 4. model.bind_tools([...]) ile üç tool'un tamamını model'e bind edin

# 5. Farklı queries ile test edin ve LLM'in hangi tool'u seçtiğini observe edin

# 6. Tool functions'ı name'e göre bulmak için tools_map dict oluşturun
#    Şununla execute edin: tool_fn.invoke(tool_call["args"])

**Additional Feature** (Optional):

Aşağıdaki durumlarda helpful error messages döndüren error handling ekleyin:

- Invalid currency code provided
- Unknown city name
- Invalid input format

**Example Output:**


**Query:** "Convert 50 EUR to JPY"
```
→ LLM chose: currency_converter
→ Args: { "amount": 50, "from": "EUR", "to": "JPY" }
→ Result: "50 EUR equals approximately 8,100 JPY"
```

**Query:** "What's the distance from Paris to Rome?"
```
→ LLM chose: distance_calculator
→ Args: { "from": "Paris", "to": "Rome", "units": "kilometers" }
→ Result: "The distance from Paris to Rome is approximately 1,430 kilometers"
```

